# Same-String Balanced Feasibility Pilot v2: Colab Execution

This is a thin, resumable launcher for the repository CLI. It does not reimplement matching, scoring, bootstrap, or endpoint logic. The protected result remains closed until two independent raters, any required adjudication, the pilot gate, and the manifest audit have passed.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive, userdata

PINNED_REPO_COMMIT = "a6f6ea10da34d55bea37355a91508e9982cdd895"
REPO_URL = "https://github.com/Fredo220/Answerability-x-Familarity-.git"
CHECKOUT = Path("/content/Answerability-x-Familarity-")
drive.mount("/content/drive")
ARTIFACT_ROOT = Path("/content/fa-same-string-feasibility-v2-artifacts")
DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/fa-same-string-feasibility-v2/checkpoints")
STATE_ROOT = ARTIFACT_ROOT / "notebook_state"

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ.get("HF_TOKEN"), "Add HF_TOKEN to Colab Secrets."


In [ ]:
def shell(command: str, *, cwd: Path | None = None) -> None:
    subprocess.run(["bash", "-lc", command], cwd=cwd, check=True)

if not CHECKOUT.exists():
    shell(f"git clone {REPO_URL} {CHECKOUT}")
shell("git fetch --all --tags --prune", cwd=CHECKOUT)
shell(f"git checkout --detach {PINNED_REPO_COMMIT}", cwd=CHECKOUT)
shell("test -z \"$(git status --porcelain --untracked-files=all)\"", cwd=CHECKOUT)
shell("python -m pip install -r requirements/fa-core.lock", cwd=CHECKOUT)
shell("python -m pip install --no-deps -e .", cwd=CHECKOUT)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
SOURCE_CONFIG = CHECKOUT / "configs/familiarity_answerability_same_string_gemma2_2b.json"
SAME_STRING_CONFIG = CHECKOUT / "configs/familiarity_answerability_same_string_feasibility_v2.json"
SMOKE_CONFIG = CHECKOUT / "configs/familiarity_answerability_qwen17b_smoke.json"
CHECKED_SOURCE_ROOT = CHECKOUT / "data/fa/confirmatory_source_v5"
CHECKED_RATINGS_ROOT = CHECKOUT / "data/fa/human_ratings/same_string_primary_v1"
SOURCE_ROOT = ARTIFACT_ROOT / "inputs/confirmatory_source_v5"
SMOKE_INPUT_ROOT = ARTIFACT_ROOT / "inputs/pilot"

from trajectory_extractor.fa_colab_snapshot import VerifiedColabSnapshotStore

def digest(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

SNAPSHOTS = VerifiedColabSnapshotStore(
    artifact_root=ARTIFACT_ROOT,
    checkpoint_root=DRIVE_CHECKPOINT_ROOT,
    scratch_root="/content",
)
SNAPSHOTS.restore_latest()
STATE_ROOT.mkdir(parents=True, exist_ok=True)

def checkpoint() -> Path:
    return SNAPSHOTS.checkpoint()

def run_fa(command: str, config: Path, *arguments: str, durable_interval_seconds: int | None = None, allow_infrastructure_failure: bool = False) -> dict:
    process = subprocess.Popen(
        ["feature-dynamics", command, "--config", str(config), "--root", str(ARTIFACT_ROOT), *arguments],
        cwd=CHECKOUT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    while True:
        try:
            stdout, stderr = process.communicate(timeout=durable_interval_seconds)
            break
        except subprocess.TimeoutExpired:
            checkpoint()
    if stderr.strip():
        print(stderr)
    payload = json.loads(stdout.strip().splitlines()[-1])
    print(json.dumps(payload, indent=2, sort_keys=True))
    checkpoint()
    if process.returncode != 0 and not (allow_infrastructure_failure and payload.get("status") == "infrastructure_failure"):
        raise RuntimeError(f"CLI transaction failed: {payload}")
    return payload

def copy_verified(source: Path, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert digest(destination) == digest(source), f"Source drift: {destination}"
    else:
        shutil.copy2(source, destination)
    assert digest(destination) == digest(source)
    return destination

def load_state(name: str) -> dict | None:
    path = STATE_ROOT / f"{name}.json"
    return json.loads(path.read_text()) if path.exists() else None

def save_state(name: str, payload: dict) -> dict:
    path = STATE_ROOT / f"{name}.json"
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True))
    temporary.replace(path)
    checkpoint()
    return payload


## 1. Reproduce the immutable v1 source matches

Copy the checked-in sources into the local atomic artifact root, verify their hashes, and deterministically select the registered 64/32/48/24/24 units. Content-addressed checkpoints mirror completed state to Drive. This path does not use the failed R11 familiarity screen.


In [ ]:
candidate_paths = [copy_verified(path, SOURCE_ROOT / path.name) for path in sorted(CHECKED_SOURCE_ROOT.glob("candidate_entities_*_v1.json"))]
synthetic_paths = [copy_verified(path, SOURCE_ROOT / path.name) for path in sorted(CHECKED_SOURCE_ROOT.glob("synthetic_candidates_*_v1.json"))]
assert len(candidate_paths) == len(synthetic_paths) == 5
source_matches = load_state("v1_same_string_matches")
if source_matches is None:
    match_args = []
    for path in candidate_paths:
        match_args.extend(["--candidate-manifest", str(path)])
    for path in synthetic_paths:
        match_args.extend(["--synthetic-manifest", str(path)])
    source_matches = save_state("v1_same_string_matches", run_fa("fa-prepare-same-string-matches", SOURCE_CONFIG, *match_args, "--shard-id", "same-string-matches-v1"))
assert source_matches["split_counts"] == {"mechanism_train": 64, "locked_validation": 32, "behavior_test": 48, "probe_test": 24, "intervention_test": 24}
SOURCE_MATCH_MANIFEST = Path(source_matches["manifest"])


## 2. Reproduce the completed blind v1 naturalness audit

Issue packets to exactly two independent raters. Stop after issuance, distribute only the blinded public packet files, and collect their responses independently. A third independent adjudicator is used only for reported disagreements.


In [ ]:
packets = load_state("naturalness_issuance")
if packets is None:
    packets = save_state("naturalness_issuance", run_fa(
        "fa-prepare-naturalness-ratings", SOURCE_CONFIG,
        "--matches-manifest", str(SOURCE_MATCH_MANIFEST),
        "--output-dir", str(ARTIFACT_ROOT / "rater-packets"),
        "--rater-id", "rater-a", "--rater-id", "rater-b",
        "--shard-id", "same-string-naturalness-issuance-v1",
    ))
print("Blinded packet issuance verified.")


In [ ]:
RATER_RESPONSE_PATHS = [
    CHECKED_RATINGS_ROOT / "rater-a-response.csv",
    CHECKED_RATINGS_ROOT / "rater-b-response.csv",
]
RATER_RESPONSE_SHA256S = {
    "rater-a-response.csv": "aed43152d66555d3546bf24ced1ea0f075ed2accfc4378050d6d1ff1ed773614",
    "rater-b-response.csv": "bee9d6ad5f6fef140264450dccaf5869634dd077c04670aa6dfda60d2220efa8",
    "rater-c-response.csv": "d1b26fb6681a3d08872545d1c45b978880ad663c1c34d5a0e1a06ae0296b8784",
}
assert all(
    path.exists() and digest(path) == RATER_RESPONSE_SHA256S[path.name]
    for path in (*RATER_RESPONSE_PATHS, CHECKED_RATINGS_ROOT / "rater-c-response.csv")
)
ratings = load_state("naturalness_ratings")
if ratings is None:
    assert len(RATER_RESPONSE_PATHS) == 2 and all(path.exists() for path in RATER_RESPONSE_PATHS)
    compile_args = []
    for path in RATER_RESPONSE_PATHS:
        compile_args.extend(["--response", str(path)])
    ratings = save_state("naturalness_ratings", run_fa(
        "fa-compile-naturalness-ratings", SOURCE_CONFIG,
        "--matches-manifest", str(SOURCE_MATCH_MANIFEST),
        "--issuance-manifest", packets["issuance_manifest"],
        *compile_args, "--shard-id", "same-string-naturalness-ratings-v1",
        "--adjudicator-id", "rater-c",
        "--adjudication-output-dir", str(ARTIFACT_ROOT / "adjudication-packet"),
    ))
print("Naturalness status:", ratings["status"])


In [ ]:
ratings = load_state("naturalness_ratings")
assert ratings is not None
if ratings["status"] == "needs_adjudication":
    adjudication_response = CHECKED_RATINGS_ROOT / "rater-c-response.csv"
    assert adjudication_response.exists()
    ratings = save_state("naturalness_ratings", run_fa(
        "fa-finalize-naturalness-adjudication", SOURCE_CONFIG,
        "--matches-manifest", str(SOURCE_MATCH_MANIFEST),
        "--initial-submission-manifest", ratings["initial_submission_manifest"],
        "--adjudication-issuance-manifest", ratings["adjudication_issuance_manifest"],
        "--adjudication-response", str(adjudication_response),
        "--shard-id", "same-string-naturalness-final-v1",
    ))
assert ratings["status"] == "compiled"
assert ratings["naturalness_gate"]["status"] == "failed"
SOURCE_RATINGS_MANIFEST = Path(ratings["ratings_manifest"])
v2_matches = load_state("v2_same_string_matches")
if v2_matches is None:
    v2_matches = save_state("v2_same_string_matches", run_fa(
        "fa-prepare-same-string-v2-matches", SAME_STRING_CONFIG,
        "--source-config", str(SOURCE_CONFIG),
        "--source-matches-manifest", str(SOURCE_MATCH_MANIFEST),
        "--source-naturalness-ratings-manifest", str(SOURCE_RATINGS_MANIFEST),
        "--shard-id", "same-string-feasibility-v2",
    ))
assert v2_matches["split_counts"] == {"behavior_test": 32, "mechanism_train": 12, "locked_validation": 4, "probe_test": 4}
MATCH_MANIFEST = Path(v2_matches["manifest"])
RATINGS_MANIFEST = SOURCE_RATINGS_MANIFEST


## 3. Unprotected Qwen runtime smoke

If no verified smoke gate already exists in Drive, build it from the checked-in pilot inputs with `fa-run-screening`, `fa-screen-entities`, `fa-build-pilot`, `fa-run-generation`, and `fa-score-behavior`. This development-only gate checks runtime viability and is not empirical evidence for the Same-String hypothesis.


In [ ]:
pilot_sources = CHECKOUT / "data/fa/pilot_inputs"
smoke_candidates = copy_verified(pilot_sources / "candidates_v4.json", SMOKE_INPUT_ROOT / "candidates_v4.json")
smoke_questions = copy_verified(pilot_sources / "screening_questions_v4.json", SMOKE_INPUT_ROOT / "screening_questions_v4.json")
smoke_synthetics = copy_verified(pilot_sources / "synthetic_candidates_v5.json", SMOKE_INPUT_ROOT / "synthetic_candidates_v5.json")
smoke_gate = load_state("smoke_gate")
if smoke_gate is None:
    screening = load_state("smoke_screening")
    if screening is None:
        screening_shard_id = os.environ.get("FA_SMOKE_SCREENING_SHARD_ID", "same-string-prerequisite-screening-v1")
        screening = run_fa("fa-run-screening", SMOKE_CONFIG, "--candidates-manifest", str(smoke_candidates), "--questions-manifest", str(smoke_questions), "--shard-id", screening_shard_id, "--namespace", "pilot", durable_interval_seconds=60, allow_infrastructure_failure=True)
        assert screening["status"] == "generated", "Preserve the failure artifact, set a new FA_SMOKE_SCREENING_SHARD_ID, and rerun."
        save_state("smoke_screening", screening)
    smoke_matches = load_state("smoke_matches")
    if smoke_matches is None:
        smoke_matches = save_state("smoke_matches", run_fa("fa-screen-entities", SMOKE_CONFIG, "--candidates-manifest", str(smoke_candidates), "--questions-manifest", str(smoke_questions), "--screening-manifest", screening["shard_manifest"], "--synthetic-manifest", str(smoke_synthetics)))
    smoke_manifest = load_state("smoke_prompt_manifest")
    if smoke_manifest is None:
        smoke_manifest = save_state("smoke_prompt_manifest", run_fa("fa-build-pilot", SMOKE_CONFIG, "--matches-manifest", smoke_matches["manifest"]))
    smoke_generation = load_state("smoke_generation")
    if smoke_generation is None:
        smoke_generation = run_fa("fa-run-generation", SMOKE_CONFIG, "--manifest", smoke_manifest["manifest"], "--shard-id", "same-string-prerequisite-smoke", "--namespace", "pilot", "--resume", durable_interval_seconds=60, allow_infrastructure_failure=True)
        assert smoke_generation["status"] == "generated", "Rerun this cell to resume the same unprotected generation shard."
        save_state("smoke_generation", smoke_generation)
    smoke_gate = save_state("smoke_gate", run_fa("fa-score-behavior", SMOKE_CONFIG, "--manifest", smoke_manifest["manifest"], "--generation-manifest", smoke_generation["shard_manifest"]))
assert smoke_gate["pilot_gate"]["status"] == "passed"
PILOT_GATE_MANIFEST = Path(smoke_gate["pilot_gate_manifest"])


## 4. Build and audit the confirmatory index

The returned `manifest` is a `same_string_confirmatory_index`, not an individual prompt capability. The CLI verifies every bound capability, hash, rating, assignment, and typed seal.


In [ ]:
confirmatory = load_state("confirmatory_index")
if confirmatory is None:
    confirmatory = save_state("confirmatory_index", run_fa(
        "fa-build-same-string-confirmatory", SAME_STRING_CONFIG,
        "--matches-manifest", str(MATCH_MANIFEST),
        "--pilot-gate-manifest", str(PILOT_GATE_MANIFEST),
        "--naturalness-ratings-manifest", str(RATINGS_MANIFEST),
    ))
CONFIRMATORY_INDEX = Path(confirmatory["manifest"])
audit = run_fa("fa-audit-manifest", SAME_STRING_CONFIG, "--manifest", str(CONFIRMATORY_INDEX))
assert audit["status"] == "passed"


## 5. One-shot protected behavior endpoint

Set the guard only after reviewing every prior artifact. The CLI writes atomically to local Colab storage while periodic content-addressed archives mirror resumable state to Drive. The sealed capability is `behavior_test`; never create a replacement protected endpoint.


In [ ]:
OPEN_PROTECTED_ENDPOINT = False
assert OPEN_PROTECTED_ENDPOINT, "Set True only after human review and the manifest audit pass."
sealed = load_state("behavior_seal")
if sealed is None:
    sealed = save_state("behavior_seal", run_fa("fa-seal-behavior-test", SAME_STRING_CONFIG, "--behavior-test-manifest", str(CONFIRMATORY_INDEX)))


In [ ]:
assert OPEN_PROTECTED_ENDPOINT and sealed is not None
result = load_state("behavior_result")
if result is None:
    result = save_state("behavior_result", run_fa("fa-evaluate-behavior-test", SAME_STRING_CONFIG, "--manifest", str(CONFIRMATORY_INDEX), "--shard-id", "same-string-feasibility-behavior-v2", durable_interval_seconds=60))
assert result["endpoint_state"] == "closed"
result


## Resume and export

On a new runtime, rerun setup and then the cells in order. The newest hash-valid Drive archive is restored to local storage; state files skip completed commands, while each CLI consumer re-verifies artifact checksums and lineage. If protected evaluation was interrupted after unlock, rerun the final evaluation cell with the same index and shard ID. If the endpoint is already `evaluated`, the CLI verifies its lineage and closes it without loading the model. Download the final metrics manifest and preserve the complete checkpoint set for local reporting.
